In [1]:
# ACC102 Track 2: US Stock Return Analysis (2022)
# Author: Jingyi Guan | Student ID: 2473524
# Data Source: WRDS CRSP | Date Accessed: 2026-04-24


# 1. Setup & Connect to WRDS

import wrds
import pandas as pd
username = "jingyiguan"
db = wrds.Connection(wrds_username=username)

Loading library list...
Done


In [2]:
# 2. Define Parameters
ticker_list = ['NVDA', 'MSFT', "AAPL"]
start_date = "2022-01-01"
end_date = "2022-12-31"

# 3. SQL Query
selected_columns = """
b.htsymbol AS ticker,
a.date,
a.permno,
a.prc,
a.ret
"""

ticker_str = "', '".join(ticker_list)


sql_query = f"""
SELECT {selected_columns}
FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol IN ('{ticker_str}')
AND a.date >= '{start_date}'
AND a.date <= '{end_date}'
"""



print(sql_query)

# 4. Load & Clean Data
df = db.raw_sql(sql_query)

print(df.head())



SELECT 
b.htsymbol AS ticker,
a.date,
a.permno,
a.prc,
a.ret

FROM crsp.msf AS a
LEFT JOIN crsp.msfhdr AS b
    ON a.permno = b.permno
WHERE b.htsymbol IN ('NVDA', 'MSFT', 'AAPL')
AND a.date >= '2022-01-01'
AND a.date <= '2022-12-31'

  ticker        date  permno        prc       ret
0   MSFT  2022-01-31   10107  310.98001 -0.075345
1   MSFT  2022-02-28   10107  298.79001 -0.037205
2   MSFT  2022-03-31   10107     308.31  0.031862
3   MSFT  2022-04-29   10107  277.51999 -0.099867
4   MSFT  2022-05-31   10107     271.87 -0.018125


In [3]:
stock_joined = db.raw_sql(sql_query, date_cols=["date"])
stock_joined

,ticker,date,permno,prc,ret
0,MSFT,2022-01-31,10107,310.98001,-0.075345
1,MSFT,2022-02-28,10107,298.79001,-0.037205
2,MSFT,2022-03-31,10107,308.31,0.031862
3,MSFT,2022-04-29,10107,277.51999,-0.099867
4,MSFT,2022-05-31,10107,271.87,-0.018125
5,MSFT,2022-06-30,10107,256.82999,-0.055321
6,MSFT,2022-07-29,10107,280.73999,0.093097
7,MSFT,2022-08-31,10107,261.47,-0.066432
8,MSFT,2022-09-30,10107,232.89999,-0.109267
9,MSFT,2022-10-31,10107,232.13,-0.003306


In [4]:
# Rename columns
stock_joined_renamed = stock_joined.rename(columns={
    "prc": "price",
    "ret": "monthly_return"
})

stock_joined_renamed.head()


,ticker,date,permno,price,monthly_return
0,MSFT,2022-01-31,10107,310.98001,-0.075345
1,MSFT,2022-02-28,10107,298.79001,-0.037205
2,MSFT,2022-03-31,10107,308.31,0.031862
3,MSFT,2022-04-29,10107,277.51999,-0.099867
4,MSFT,2022-05-31,10107,271.87,-0.018125


In [5]:
# Sort by date
stock_joined_sorted = stock_joined_renamed.sort_values(by="date")
stock_joined_sorted.head()

,ticker,date,permno,price,monthly_return
0,MSFT,2022-01-31,10107,310.98001,-0.075345
24,NVDA,2022-01-31,86580,244.86,-0.167454
12,AAPL,2022-01-31,14593,174.78,-0.015712
1,MSFT,2022-02-28,10107,298.79001,-0.037205
25,NVDA,2022-02-28,86580,243.85001,-0.004125


In [6]:
# Extract year/month/day
stock_joined_sorted["year"] = stock_joined_sorted["date"].dt.year
stock_joined_sorted["month"] = stock_joined_sorted["date"].dt.month
stock_joined_sorted["day"] = stock_joined_sorted["date"].dt.day
stock_joined_sorted.head()


,ticker,date,permno,price,monthly_return,year,month,day
0,MSFT,2022-01-31,10107,310.98001,-0.075345,2022,1,31
24,NVDA,2022-01-31,86580,244.86,-0.167454,2022,1,31
12,AAPL,2022-01-31,14593,174.78,-0.015712,2022,1,31
1,MSFT,2022-02-28,10107,298.79001,-0.037205,2022,2,28
25,NVDA,2022-02-28,86580,243.85001,-0.004125,2022,2,28


In [7]:
# 5. Descriptive Analysis
# Top returns
stock_by_return = stock_joined_sorted[stock_joined_sorted["monthly_return"].notna()].sort_values(by="monthly_return", ascending=False)
stock_by_return.head(10)

,ticker,date,permno,price,monthly_return,year,month,day
34,NVDA,2022-11-30,86580,169.23,0.254131,2022,11,30
30,NVDA,2022-07-29,86580,181.63,0.198166,2022,7,29
18,AAPL,2022-07-29,14593,162.50999,0.188634,2022,7,29
26,NVDA,2022-03-31,86580,272.85999,0.119131,2022,3,31
33,NVDA,2022-10-31,86580,134.97,0.111871,2022,10,31
21,AAPL,2022-10-31,14593,153.34,0.109551,2022,10,31
10,MSFT,2022-11-30,10107,255.14,0.102055,2022,11,30
6,MSFT,2022-07-29,10107,280.73999,0.093097,2022,7,29
14,AAPL,2022-03-31,14593,174.61,0.057473,2022,3,31
2,MSFT,2022-03-31,10107,308.31,0.031862,2022,3,31


In [8]:
# Summary statistics by ticker
summary_stats = stock_joined_sorted.groupby("ticker")["monthly_return"].agg(
    mean_return="mean",
    std_risk="std",
    min_return="min",
    max_return="max",
    count="count"
).round(4)

summary_stats


,mean_return,std_risk,min_return,max_return,count
ticker,,,,,
AAPL,-0.0212,0.095,-0.1223,0.1886,12
MSFT,-0.0248,0.0695,-0.1093,0.1021,12
NVDA,-0.0409,0.1814,-0.3203,0.2541,12


In [9]:
# 6. Reusable Function
def get_monthly_stock_data(username, htsymbol, start_date, limit=120):
    import wrds
    selected_columns = """
        b.htsymbol,
        b.permno,
        a.date,
        a.prc,
        a.ret
    """
    sql_query = f"""
    SELECT {selected_columns}
    FROM crsp.msf AS a
    LEFT JOIN crsp.msfhdr AS b
        ON a.permno = b.permno
    WHERE b.htsymbol = '{htsymbol}'
    AND a.date >= '{start_date}'
    LIMIT {limit}
    """
    db = wrds.Connection(wrds_username=username)
    df = db.raw_sql(sql_query, date_cols=["date"])
    df = df.rename(columns={"prc": "price", "ret": "monthly_return"})
    df = df.sort_values(by="date")
    df["year"] = df["date"].dt.year
    df["month"] = df["date"].dt.month
    df["day"] = df["date"].dt.day
    db.close()
    return df

# Test function
nvda_stock = get_monthly_stock_data(username, "NVDA", "2022-01-01", limit=36)
nvda_stock


Loading library list...
Done


,htsymbol,permno,date,price,monthly_return,year,month,day
0,NVDA,86580,2022-01-31,244.86,-0.167454,2022,1,31
1,NVDA,86580,2022-02-28,243.85001,-0.004125,2022,2,28
2,NVDA,86580,2022-03-31,272.85999,0.119131,2022,3,31
3,NVDA,86580,2022-04-29,185.47,-0.320274,2022,4,29
4,NVDA,86580,2022-05-31,186.72,0.00674,2022,5,31
5,NVDA,86580,2022-06-30,151.59,-0.187928,2022,6,30
6,NVDA,86580,2022-07-29,181.63,0.198166,2022,7,29
7,NVDA,86580,2022-08-31,150.94,-0.16897,2022,8,31
8,NVDA,86580,2022-09-30,121.39,-0.195508,2022,9,30
9,NVDA,86580,2022-10-31,134.97,0.111871,2022,10,31


In [10]:
# 7. Close Connection
db.close()